# Music4All-Onion Preprocessing

Dataset: https://zenodo.org/records/6609677

- Restrict music listening events to those occurring in **2019**
- Consider only user–track interactions with **play count ≥ 2**

In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv('../.env')

# Use DATA_ROOT from .env if set, otherwise relative path
data_root = os.getenv('DATA_ROOT', os.getenv('DATASET_PATH', '../dataset'))
onion_dir = os.path.join(data_root, 'onion')

# Prefer parquet (faster for large data), fallback to tsv
parquet_path = os.path.join(onion_dir, 'userid_trackid_timestamp_full.parquet')
tsv_path = os.path.join(onion_dir, 'userid_trackid_timestamp.tsv.bz2')
if not os.path.exists(tsv_path):
    tsv_path = os.path.join(onion_dir, 'userid_trackid_timestamp.tsv')  # uncompressed

use_parquet = os.path.exists(parquet_path)
data_path = parquet_path if use_parquet else tsv_path
print(f'Data root: {data_root}')
print(f'Onion dir: {onion_dir}')
print(f'Using parquet: {use_parquet}')
print(f'Data path: {data_path}')

In [14]:
# Load listening events: userid, trackid, timestamp
print('Loading userid_trackid_timestamp...')
if use_parquet:
    events = pd.read_parquet(data_path)
    # Map column names to user, item, timestamp (parquet may use userid, trackid)
    col_map = {'userid': 'user', 'trackid': 'item', 'user_id': 'user', 'track_id': 'item'}
    events = events.rename(columns={c: col_map[c] for c in col_map if c in events.columns})
    if 'timestamp' not in events.columns and 'ts' in events.columns:
        events = events.rename(columns={'ts': 'timestamp'})
else:
    events = pd.read_csv(data_path, sep='\t', names=['user', 'item', 'timestamp'], header=0, low_memory=False)
print(f'Total events: {len(events):,}')
events.head(10)

Loading userid_trackid_timestamp...
Total events: 252,984,396


,user,item,timestamp
0,51549,iJTBIGHPjgJcT4Bt,2013-01-27 21:42:38
1,51549,iJTBIGHPjgJcT4Bt,2013-01-27 21:38:53
2,51549,iJTBIGHPjgJcT4Bt,2013-01-27 21:35:08
3,51549,iJTBIGHPjgJcT4Bt,2013-01-27 21:31:23
4,51549,iJTBIGHPjgJcT4Bt,2013-01-27 21:27:38
5,51549,iJTBIGHPjgJcT4Bt,2013-01-27 21:23:53
6,51549,iJTBIGHPjgJcT4Bt,2013-01-27 21:20:09
7,51549,iJTBIGHPjgJcT4Bt,2013-01-27 21:16:23
8,51549,iJTBIGHPjgJcT4Bt,2013-01-27 21:12:39
9,51549,iJTBIGHPjgJcT4Bt,2013-01-27 21:08:54


In [4]:
import pandas as pd


events["timestamp"] = pd.to_datetime(events["timestamp"])

start_date = "2019-01-01"
end_date = "2019-12-31"

df_2019 = events[
    (events["timestamp"] >= start_date) &
    (events["timestamp"] <= end_date)
]

print("After time filtering:", len(df_2019))

play_counts = df_2019.groupby(["user", "item"])["item"].transform("count")

# keep play_count >= 2
df_filtered = df_2019[play_counts >= 2]

print("Final events:", len(df_filtered))

After time filtering: 16754087
Final events: 13915147


In [14]:
import pandas as pd

events_filtered = df_filtered.copy()  
events_filtered.loc[:, "datetime"] = pd.to_datetime(
    events_filtered["timestamp"], errors="coerce"
)

events_filtered.loc[:, "ts"] = (
    events_filtered["datetime"].view("int64") // 10**9
)

events_filtered.loc[events_filtered["datetime"].isna(), "ts"] = pd.NA

In [16]:
df_output = events_filtered[['user', 'item', 'ts']].copy()
df_output['rating'] = 1
df_output = df_output[['user', 'item', 'rating', 'ts']]
df_output.columns = ['user', 'item', 'rating', 'timestamp']
df_output['timestamp'] = df_output['timestamp'].astype(int)
df_output = df_output.sort_values(['user', 'timestamp'])

print(df_output.shape)
print(f'Unique users: {df_output["user"].nunique():,}')
print(f'Unique items: {df_output["item"].nunique():,}')
df_output.head(10)

(13915147, 4)
Unique users: 15,949
Unique items: 55,535


,user,item,rating,timestamp
206044628,0,7uwg0JpB7Dos5kOU,1,1546302839
206044625,0,7uwg0JpB7Dos5kOU,1,1546314796
206044547,0,t7rXCkriVK4E9GvI,1,1546316928
206044777,0,t7rXCkriVK4E9GvI,1,1546318244
206044620,0,ax6Kn8vv2I655Vnz,1,1546454139
206044545,0,ax6Kn8vv2I655Vnz,1,1546454363
206044619,0,t7rXCkriVK4E9GvI,1,1546454442
206044544,0,mTD4iHCTKzTTr7GP,1,1546540102
206044618,0,mTD4iHCTKzTTr7GP,1,1546540815
190969998,2,h5P7WshNdjpbVMzT,1,1546309294


In [12]:
# Save to data.csv
save_path = "../dataset/onion/data.csv"
df_output.to_csv(save_path, header=None, index=False)
print(f'Saved to {save_path}')

Saved to ../dataset/onion/data.csv


In [17]:
# Verify output
print(f'Final: {len(df_output):,} interactions, {df_output["user"].nunique():,} users, {df_output["item"].nunique():,} items')

Final: 13,915,147 interactions, 15,949 users, 55,535 items
